# AI(Computer Vision)기반 주유 공정 자동화 어시스턴트

## 1. 프로젝트 설명

### 👉 역할
PM(프로젝트 매니저), 로봇 팔 제어 알고리즘 개발, 공정 물리 환경 제작, GUI 제작

### 👉 개발 환경
하드웨어 - 협동로봇(Doosan m0609), PC, RealSense 카메라, 주유소 축소 모형

소프트웨어 - Ubuntu 22.04(ROS2 Humble), Python, Doosan API, YOLO, RViz2, GPT4-Langchain, STT, LLM
### 👉 프로젝트 개요
AI(Computer Vision) 기반 협동 로봇 작업 어시스턴트 구현 프로젝트.

사람이 직접 내려서 주유하는 기존의 무인 주유소 방식과 차별화 되는 완전 자동화 구현.
### 👉 프로젝트 목표
RGB-D 카메라가 장착된 협동로봇을 활용하여 STT(음성인식 및 문자열 추출), LLM(명령해석)과정에서 도출된 타겟을
 
YOLO(AI객체탐지)를 활용하여 인지후 추적하고 설계한 경로대로 로봇 팔을 안정적으로 제어.

음성인식을 통해 추출된 키워드와 경유 상태를 GUI를 통해 모니터링 가능하게 구현.

### 👉 중요성과 및 문제해결
Modbus TCP 기반 그리퍼 직접 제어, 잡는 물체별 폭과 힘 값을 튜닝해 그립 안정성 향상.

프롬프트를 통해 연료/결제/금액 정보를 고정 포맷으로 추출해 구조적으로 오류 예방.

객체 인지 실패 시, Fallback 위치와 자세로 진행하는 예외처리를 추가해 전체 시나리오 진행 유지.

### 👉 Task 환경 구성
<img src="Task.png" width="600">

### 👉 시나리오 구성
<img src="scenario.png" width="600">


## 2. 활용 장치 설명

### 그리퍼: RG2 - 와이드 스트로크가 있는 유연한 2-핑거 로봇 그리퍼
<img src="onrobot.png" width="150">

### 산업용 로봇: Doosan Robotics - M0609
<img src="doosan_corobot.png" width="300">

### 인텔® RealSense depth카메라 D435
<img src="intel_realsence.png" width="600">

<작동 사양>

작동 범위(최소-최대): ~.3m - 3m

뎁스 해상도 및 FPS: 1280 X 720

뎁스 시야: 85.2 x 58

## 3. 제어 코드

### System Architecture Diagram
<img src="SysArch_diagram.png" width="800">

### ROS 실행 명령어


| 플랫폼 | 코드 | 기능 |
| --- | --- | --- |
| 협동로봇 | ros2 launch dsr_bringup2 dsr_bringup2_rviz.launch.py mode:=real host:=192.168.1.100 port:=12345 model:=m0609 | 두산 협동로봇 rivz와 pc 연결 |
| 그리퍼 | [http://192.168.1.1](http://192.168.1.1/#/devices)(아이디: admin, 비밀번호: 12345678) | 그리퍼 설정 |
|  | ros2 run pick_and_place_voice onrobot_robot_control | 티칭 수동모드 on/off |
|  | < 최종 시현용 명령어> |  |
|  | ros2 run pick_and_place_voice detection.py" | object_detection.py 실행 |
|  | ros2 run pick_and_place_voice get_keyword(making)  | get_keyword_custom.py 실행 |
|  | ros2 run pick_and_place_voice keyword_extraction_3  | getword_extraction_3.py 실행 |
|  | ros2 run pick_and_place_voice gas_station_yolo_stt_mapping | gas_station_yolo_stt_mapping.py 실행 |
|  | ros2 run pick_and_place_voice onrobot_robot_control | onrobot_robot_control_7.py 실행  |

### A. Modubus TCP 기반 OnRobot RG 그리퍼 제어 코드

설명: 통신 및 제어 코드 ( 하드웨어 제어용 Low - Level 인터페이스 )

In [ ]:
#!/usr/bin/env python3
# ========================== Gripper Class ==========================
from pymodbus.client.sync import ModbusTcpClient as ModbusClient

class RG(): # OnRobot RG 그리퍼 제어 class
    # Modubus TCP 클라이언트 생성 및 초기 설정(그리퍼 타입, 최대 폭과 힘)
    def __init__(self, gripper, ip, port): 
        self.client = ModbusClient(
            ip,
            port=port,
            stopbits=1,
            bytesize=8,
            parity='E',
            baudrate=115200,
            timeout=1)
        if gripper not in ['rg2', 'rg6']:
            print("Please specify either rg2 or rg6.")
            return
        self.gripper = gripper
        if self.gripper == 'rg2':
            self.max_width = 500
            self.max_force = 400
        elif self.gripper == 'rg6':
            self.max_width = 1600
            self.max_force = 1200
        self.open_connection()
    
    def open_connection(self): #  Modbus 서버 연결
        self.client.connect()

    def close_connection(self): # #  Modbus 연결 종료
        self.client.close()

    def get_fingertip_offset(self): # 그리퍼 offset(mm) 읽고 설정
        result = self.client.read_holding_registers(address=258, count=1, unit=65)
        offset_mm = result.registers[0] / 10.0
        return offset_mm

    def get_width(self): # 현재 그리퍼 open 거리값(mm) 반환
        result = self.client.read_holding_registers(address=267, count=1, unit=65)
        width_mm = result.registers[0] / 10.0
        return width_mm

    # 상태 레지스터 기반 동작 여부, 그립 감지, 안전 스위치 상태 등 비트 단위 반환
    def get_status(self):
        result = self.client.read_holding_registers(address=268, count=1, unit=65)
        status = format(result.registers[0], '016b')
        status_list = [0] * 7
        if int(status[-1]):
            print("A motion is ongoing so new commands are not accepted.")
            status_list[0] = 1
        if int(status[-2]):
            print("An internal- or external grip is detected.")
            status_list[1] = 1
        if int(status[-3]):
            print("Safety switch 1 is pushed.")
            status_list[2] = 1
        if int(status[-4]):
            print("Safety circuit 1 is activated so it will not move.")
            status_list[3] = 1
        if int(status[-5]):
            print("Safety switch 2 is pushed.")
            status_list[4] = 1
        if int(status[-6]):
            print("Safety circuit 2 is activated so it will not move.")
            status_list[5] = 1
        if int(status[-7]):
            print("Any of the safety switch is pushed.")
            status_list[6] = 1
        return status_list
    
    # 오프셋 반영 실제 그리퍼 open 거리값(mm) 반환
    def get_width_with_offset(self):
        result = self.client.read_holding_registers(address=275, count=1, unit=65)
        width_mm = result.registers[0] / 10.0
        return width_mm

    # 그리퍼 제어 모드 설정(레지스터값 기반 제어 방식 지정)
    def set_control_mode(self, command):
        result = self.client.write_register(address=2, value=command, unit=65)
    def set_target_force(self, force_val): # 힘 설정
        result = self.client.write_register(address=0, value=force_val, unit=65)
    def set_target_width(self, width_val): # 폭 설정
        result = self.client.write_register(address=1, value=width_val, unit=65)
    def close_gripper(self, force_val=400): # 그리퍼 close
        params = [force_val, 0, 16]
        mwait()
        print("Start closing gripper.")
        result = self.client.write_registers(address=0, values=params, unit=65)
    def open_gripper(self, force_val=400): # 그리퍼 open
        params = [force_val, self.max_width, 16]
        print("Start opening gripper.")
        result = self.client.write_registers(address=0, values=params, unit=65)
    
    # 추가한 함수
    def open_gripper2(self, force_val=400): # 그리퍼 open 값2
        params = [force_val, self.max_width//2, 16]
        print("Start opening gripper.")
        result = self.client.write_registers(address=0, values=params, unit=65)
    def move_gripper(self, width_val, force_val=400): # 그리퍼 제어 함수
        params = [force_val, width_val, 16]
        print("Start moving gripper.")
        result = self.client.write_registers(address=0, values=params, unit=65)


### B. 협동로봇 시나리오 별 이동 제어 코드

설명: 음성 인식과 객체 인식을 기반으로 협동로봇이 자동으로 주유 시나리오를 수행하도록 제어 구조 설계, YOLO 기반 객체 인식을 활용해 주유기와 카드 위치를 인식하고 해당 위치로 로봇을 제어하여 주유기, 결제, 차단기 on/off까지 수행하게 설계

In [ ]:
# ========================== Robot Control Class ==========================
import sys, os, time
import numpy as np
from scipy.spatial.transform import Rotation
import rclpy
from rclpy.node import Node
import DR_init
from std_msgs.msg import String, Int32, Bool
# from od_msg.srv import SrvDepthPosition
from std_srvs.srv import Trigger
from ament_index_python.packages import get_package_share_directory

# Initialize Robot and Gripper
package_path = get_package_share_directory("pick_and_place_voice")
CALIBRATION_NAME = "T_gripper2camera.npy"
CALIBRATION_PATH = os.path.join(package_path, "resource", CALIBRATION_NAME)

ROBOT_ID, ROBOT_MODEL = "dsr01", "m0609"
VELOCITY, ACC = 40, 40
GRIPPER_NAME, TOOLCHARGER_IP, TOOLCHARGER_PORT = "rg2", "192.168.1.1", 502

DR_init.__dsr__id, DR_init.__dsr__model = ROBOT_ID, ROBOT_MODEL
rclpy.init()
dsr_node = rclpy.create_node("robot_control_node", namespace=ROBOT_ID)
DR_init.__dsr__node = dsr_node

from DSR_ROBOT2 import movej, movel, mwait, posx, DR_BASE, DR_MV_MOD_REL, check_force_condition, DR_TOOL, DR_AXIS_Z, get_current_posx
gripper = RG(GRIPPER_NAME, TOOLCHARGER_IP, TOOLCHARGER_PORT)

class AutoFuelControlNode(Node):
    def __init__(self): # node 초기화, 로봇 위치 초기화
        super().__init__("auto_fuel_control")
        self.fuel_type = None
        self.detected_object = None
        self.stt_success = False
        self.detected_bbox = None

        self.yolo_sub = self.create_subscription(String, 'yolo_result', self.yolo_callback, 10)
        self.fuel_sub = self.create_subscription(Int32, '/oil', self.fuel_callback, 10)
        self.card_sub = self.create_subscription(Bool, '/target/CreditCard', self.card_callback, 10)
        self.status_pub = self.create_publisher(String, '/robot/status', 10)

        self.get_logger().info("자동 주유 로봇 제어 노드 시작됨.")
        self.init_robot()
        self.get_logger().info("로봇 위치 초기화 완료")
        # self.wait_for_nudge_then_comeback()
        
        # ─ 좌표 변환 유틸 준비
        # self.gripper2cam_path = CALIBRATION_PATH
        # self.tf = Transformation(self.gripper2cam_path)

    def publish_status(self, message): # Topic 상태 msg 발행
        msg = String(); msg.data = message
        self.status_pub.publish(msg)
        self.get_logger().info(f"[STATUS] {message}")

    def init_robot(self): # 로봇 초기 자세 이동
        movej([0, 0, 90, 0, 90, 0], vel=VELOCITY, acc=ACC)
        gripper.open_gripper()
        mwait()

# ================= YOLO 콜백 =================
    def _get_depth(self, x, y): # 특정 pixel depth 값 가져오기
        frame = self._wait_for_valid_data(self.img_node.get_depth_frame, "depth frame")
        try:
            return float(frame[y, x])
        except IndexError:
            self.get_logger().warn(f"Coordinates ({x},{y}) out of range.")
            return None

    def _wait_for_valid_data(self, getter, description): # 반복 대기
        data = getter()
        while data is None or (isinstance(data, np.ndarray) and not data.any()):
            rclpy.spin_once(self.img_node)
            self.get_logger().info(f"Retry getting {description}.")
            data = getter()
        return data

    def _pixel_to_camera_coords(self, x, y, z): # pixel좌표 -> camera좌표 변환
        fx = self.intrinsics['fx']
        fy = self.intrinsics['fy']
        ppx = self.intrinsics['ppx']
        ppy = self.intrinsics['ppy']
        return (
            (x - ppx) * z / fx,
            (y - ppy) * z / fy,
            z
        )

    # 객체 탐지 정확도confidence 0.5 이상만 처리
    # pump 감지 → pump_moving() 실행
    # creditcard 감지 → card_moving() 실행
    # bbox 저장 (self.detected_bbox)
    def yolo_callback(self, msg):
        self.get_logger().info("[YOLO] STT 명령이 success.")
        # if self.stt_success: 
        #     return
        
        try:
            parts = msg.data.split()
            if len(parts) != 5:
                self.get_logger().warn(f"YOLO 메시지 형식 오류: {msg.data}") 
                return
            
            class_name, conf = parts[0].lower(), float(parts[1])
            cx, cy, cz = map(int, parts[2:])

            # if class_name == "pump": class_name = "creditcard"
            self.get_logger().info(f"[YOLO] {class_name} 감지됨 (conf={conf:.2f})")

            if conf < 0.5: 
                return
            
            self.detected_bbox = (cx, cy, cz)

            
            if class_name == "pump": 
                self.pump_moving()
                mwait()
                self.wait_for_nudge_then_comeback()

            elif class_name == "creditcard": 
                self.card_moving()
                mwait()
                self.move_stopsign()

        except Exception as e:
            self.get_logger().error(f"YOLO 콜백 예외: {e}")

    # =============== 연료 종류 콜백 ===============
    def fuel_callback(self, msg): # 연료 종류 수신 (0: 경유, 1: 휘발유)
        self.fuel_type = msg.data
        self.get_logger().info(f"[음성] 연료 인식됨: {'경유' if msg.data == 0 else '휘발유'}")
        self.execute_fuel_task()

    # 주유 전체 시퀀스: 주유구 열기(open_hole), 주유기 이동(pump_moving), 힘 감지 대기 (wait_for_nudge_then_comeback)
    def execute_fuel_task(self):
        if self.fuel_type == 0:
            self.get_logger().info("[TASK] 경유 주유 동작 실행")
        elif self.fuel_type == 1:
            self.get_logger().info("[TASK] 휘발유 주유 동작 실행")
        self.open_hole()
        mwait()
        self.pump_moving()
        mwait()
        self.wait_for_nudge_then_comeback()
        mwait()
    
    def card_callback(self, msg): # 카드 인식되면 시나리오 실행
        if msg.data:
            self.get_logger().info("[음성] 카드 결제 감지됨 → 카드 삽입 동작 실행")
            self.card_moving()
            mwait()
            self.move_stopsign()

# 0. 주유구 열기 모션
    def open_hole(self):
        # 차량 상단 위치로 이동
        movej([-19.16, 9.85, 77.59, -0.53, 92.05, -3.62], vel=VELOCITY, acc=ACC)
        gripper.close_gripper()
        mwait()
        # 경광등 누르고 올라오기
        movel(posx(0, 0, -56, 0, 0, 0), vel=100, acc=100, ref=DR_BASE, mod=DR_MV_MOD_REL)
        movel(posx(0, 0, 49, 0, 0, 0), vel=40, acc=40, ref=DR_BASE, mod=DR_MV_MOD_REL)
        self.get_logger().info(f"주유구 오픈")


# 1. 주유소에서 주유기 뽑고 주유구 까지 가는 무빙
    def pump_moving(self): # 사용자 정의(주유기가 보이는 위치)로 이동
        JOne = [24.07, 40.60, 28.48, -1.66, 111.86, -161.23]
        movej(JOne, vel=VELOCITY, acc=ACC)
        mwait()
        # X축 방향 -1cm 이동 (기준: 로봇 base 좌표계)
        movel(posx(20.0, -10.0, 50.0, 0, 0, 0), vel=VELOCITY, acc=ACC, ref=DR_BASE, mod=DR_MV_MOD_REL)

        # yolo bounding box
        if self.detected_bbox:
            robot_base = get_current_posx()[0]
            obj_pos = self.tf.obj_pose_in_base(robot_base, self.detected_bbox)
            movel(obj_pos, vel=VELOCITY, acc=ACC, ref=DR_BASE)
            time.sleep(3)
            # x1, y1, x2, y2 = self.detected_bbox
            # center_x = (x1 + x2) / 2
            # center_y = (y1 + y2) / 2

            # self.get_logger().info(f"[YOLO 기반 이동] 중심 좌표: ({center_x}, {center_y})")

            # # (카메라 좌표계 → 로봇 좌표계 변환)
            # robot_x = center_x * 0.1  # 스케일링 계수
            # robot_y = center_y * 0.1
            # robot_z = 150.0  # 고정값 또는 depth

            # target_pose = posx(robot_x, robot_y, robot_z, 0, 0, 0)
            # movel(target_pose, vel=VELOCITY, acc=ACC, ref=DR_BASE)
        else:
            self.get_logger().warn("[경고] YOLO 좌표가 없어 기본 위치로 이동합니다.")
        
        
            # # 기존 하드코딩된 JOne 사용
            # JOne = [24.07, 40.60, 28.48, -1.66, 111.86, -161.23]
            # movej(JOne, vel=VELOCITY, acc=ACC)
            # mwait()

            # 주유기 yolo detect 했을경우 가는 좌표
            JTwo = [592.74, 136.66, 183.29, 82.59, -178.16, -102.06]
            movel(JTwo, vel=VELOCITY, acc=ACC)
            mwait()

            # 주유기 잡기위한 offset
            movel(posx(0.0, 60.0, 0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
            gripper.open_gripper()
            movel(posx(0.0, 0.0, -82.0 , 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
            gripper.close_gripper()
                
            # 주유기 잡고 주유구 쪽으로 이동
            movel(posx(0.0, 0.0, 130.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
            movel(posx(0.0, -120.0, 0.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
            JThree = [7.73, 37.66, 31.52, -1.85, 111.19, -177.63]
            movej(JThree, vel=VELOCITY, acc=ACC)
            # gripper.close_gripper()
            # mwait()
            # 사용자 정의 홈 위치
            # JOne = [24.07, 40.60, 28.48, -1.66, 111.86, -161.23]
            # movej(JOne, vel=VELOCITY, acc=ACC)
            # mwait()

            JFour = [8.49, 11.99, 98.01,-7.29, 67.40,-166.85]
            movej(JFour, vel=VELOCITY, acc=ACC)

            # 주유기 주유구에 넣기
            movel(posx(0.0, -30.0, 5.0, 0, 0, 0),vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)


# 2. 주유기 주유구에서 빼고 가져다 놓기
     # 넛지 함수
    def wait_for_nudge_then_comeback(self, path_list=None, vel=100, acc=100, threshold=20.0):
            self.get_logger().info("Z축 방향으로 최소 20move.0N 이상 누르면 다음 동작을 수행합니다.")
            try:
                while rclpy.ok():
                    if not check_force_condition(axis=DR_AXIS_Z, min=threshold, ref=DR_TOOL):
                        self.get_logger().info("Nudge 감지됨! 후속 동작 수행 중")
                        break
                    time.sleep(0.1)

                movel(posx(0.0, 40.0, 0.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                # Back to place
                movel(posx(0.0, 0.0, 130.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                movel(posx(135.0, 0.0, 0.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                JTwo = [592.74, 196.66, 183.29, 82.59, -178.16, -102.06]
                movel(JTwo, vel=VELOCITY, acc=ACC)
                mwait()
                movel(posx(0.0, 0.0, -80.0 , 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
                gripper.open_gripper()

                self.card_moving()

            except Exception as e:
                print(f"예외 발생: {e}")
       

# 3. 주유구에서 카드 보이는 위치로 이동
    def card_moving(self):
        movel(posx(0.0, 0.0, 130.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)

        gripper.close_gripper()
        mwait()


        # 사용자 지정(카드가 보이는 좌표)
        JOne = [-47.17, 45.54, 39.14, 40.73, 109.0, -26.49]
        movej(JOne, vel=15, acc=15)
        mwait() 

        if self.detected_bbox:
            robot_base = get_current_posx()[0]
            obj_pos = self.tf.obj_pose_in_base(robot_base, self.detected_bbox)
            movel(obj_pos, vel=VELOCITY, acc=ACC, ref=DR_BASE)
            time.time(3)

            # x1, y1, x2, y2 = self.detected_bbox
            # cx = (x1 + x2) / 2
            # cy = (y1 + y2) / 2
            # self.get_logger().info(f"[YOLO 카드 이동] 중심 좌표: ({cx}, {cy})")

            # # 비례식 기반 간단한 변환 (실제 변환 필요시 조정)
            # robot_x = cx * 0.1
            # robot_y = cy * 0.1
            # robot_z = 103.0  # 높이는 고정 또는 조정 가능

            # movel(posx(robot_x, robot_y, robot_z, 0, 0, 0),
            #     vel=VELOCITY, acc=ACC, ref=DR_BASE)
        else:
            self.get_logger().warn("[YOLO] 감지된 카드 좌표 없음 → 기본 위치로 이동")

        # 카드 pick 좌표(movel로 이동시 급발진)
            # JTwo = [560.20, -70.72, 103.25, 65.47, 138.07, 77.18]
            JTwo = [560.20, -70.72, 103.25, 65.47, 138.07, 77.18]
            movel(JTwo, vel=VELOCITY, acc=ACC)
            mwait()

            # 신용카드와 충돌로 인한 경유점 추가
            movel(posx(-30.0, 0.0, 50.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
            mwait()

            # 카드를 잡기위한 off_set
            JThree = [-6.59, 26.58, 64.77, 0.02, 88.66, 85.44]
            movej(JThree, vel=VELOCITY, acc=ACC)

            gripper.open_gripper()
            mwait()

            movel(posx(0.0, 0.0, -60, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
            gripper.close_gripper()
            mwait()

            movel(posx(0.0, 0.0, 40.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
            
            # 그리퍼 90도 돌리기        
            JFour = [-6.64, 27.55, 69.19, 0.02, 83.26, 85.38-90]
            movej(JFour, vel=VELOCITY, acc=ACC)
            mwait()

            gripper.close_gripper()
            
            # 카드 잡고 주유기 쪽으로 이동
            JFive = [548.07, 35.66, 84.82, 86.29, -175.39, 86.74]
            movel(JFive, vel=VELOCITY, acc=ACC)
            mwait()

            # 주유기 앞에서 카드 밀어넣기 off_set
            movel(posx(0.0, 30.0, 0.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
            time.sleep(3)
            self.publish_status("카드 결제가 완료되었습니다.")
            
            # 다시 빼기
            movel(posx(0.0, -40.0, 0.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
            mwait()
            
            # 위로올리기 
            movej(JThree, vel=VELOCITY, acc=ACC)
            mwait()
            
            #내려가기
            movel(posx(0.0, 0.0, -60, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
            gripper.open_gripper2()
            mwait()
            movel(posx(0.0, 0.0, 130.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)

# pick and place stopsign
    def move_stopsign(self):
        JFive=[-14.37, -9.98, 103.61, -7.73, 77.79, -4.27]
        movej(JFive, vel=VELOCITY, acc=ACC)

        Jsix=[-21.26, 1.85, 102.93, 0.00, 75.22, -21.39]
        movej(Jsix, vel=VELOCITY, acc=ACC)
        gripper.close_gripper()
        self.publish_status("차단기가 올라가는중...")
        time.sleep(2)

        movel(posx(0.0, 0.0, 150.0, 0, 0, 0), vel=VELOCITY, acc=ACC, radius=0.0, ref=DR_BASE, mod=DR_MV_MOD_REL)
        mwait()
        self.publish_status("좋은 하루되세요.")


def main():
    node = AutoFuelControlNode()
    rclpy.spin(node)
    node.destroy_node()
    rclpy.shutdown()

if __name__ == "__main__":
    main()


협동로봇 move 관련 함수 요약

<img src="move_funtion.png" width="800">



### C. 음성인식 → STT(음성 텍스트 변환) → LLM(키워드 추출) 코드

### C-1. node 실행시 바로 동작( keyword_extraction_3.py)

특징: [자동 실행형] node 실행시 바로 동작

동작: 음성 → keyword_extraction_3 → /detected_tool (pump / creditcard) → gas_station_yolo_stt_mapping → yolo_result → robot_control

In [ ]:
import os
import rclpy
import pyaudio
from rclpy.node import Node
from std_msgs.msg import Bool, String, Int32
from dotenv import load_dotenv
from langchain_community.chat_models import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain
from voice_processing.stt import STT
from voice_processing.MicController import MicController, MicConfig
from voice_processing.wakeup_word import WakeupWord
from pathlib import Path

class ExtractKeyword:  # GPT-4o 모델 초기화, 프롬프트 정의, LLMChain 생성
    def __init__(self, api_key):
        self.llm = ChatOpenAI(
            model="gpt-4o", temperature=0.2, openai_api_key=api_key
        )

        self.prompt_template = PromptTemplate(
            input_variables=["user_input"],
            template="""
당신은 사용자 음성 명령에서 연료 종류, 결제 수단, 주유 금액을 추출해야 합니다.

<목표>
- 연료 종류: 휘발유, 경유 중 하나 (없으면 비워둠)
- 결제 수단: 카드 (없으면 비워둠)
- 금액: 숫자만 추출하여 "원" 단위로 반환 (예: "5만원" → 50000)

<출력 형식>
- 반드시 다음 형식으로 반환하세요: [연료 / 결제 / 금액]
- 각 항목이 없으면 공백으로 비워, '/' 구분자는 반드시 유지하세요

<예시>
- 입력: "경유로 5만원 카드 결제해줘" → [경유 / 카드 / 50000]
- 입력: "휘발유로 카드 결제" → [휘발유 / 카드 / ]
- 입력: "카드 결제할게" → [ / 카드 / ]

<사용자 입력>
"{user_input}"
"""
        )
        self.lang_chain = LLMChain(llm=self.llm, prompt=self.prompt_template)

    def extract(self, text): # LLM 호출, 결과 파싱, 데이터 정제
        response = self.lang_chain.invoke({"user_input": text})
        try:
            raw = response["text"].strip()
            if raw.startswith("[") and raw.endswith("]"):
                raw = raw[1:-1]  # remove brackets
            parts = raw.split("/")
            if len(parts) != 3:
                raise ValueError(f"Expected 3 fields but got: {raw}")

            fuel = parts[0].strip()
            payment = parts[1].strip()
            amount_str = parts[2].strip()

            amount = int(amount_str) if amount_str.isdigit() else 0
            return fuel, payment, amount

        except Exception as e:
            print(f"[ERROR] LLM 응답 파싱 실패: {e}")
            return "", "", 0


class ToolPublisherNode(Node): # STT, LLM, 마이크 초기화 및 퍼블리셔
    def __init__(self):
        super().__init__("tool_publisher_node")

        # env 불러오기
        dotenv_path = Path(__file__).parent.parent / "resource" / ".env"
        load_dotenv(dotenv_path=dotenv_path)
        api_key = os.getenv("OPENAI_API_KEY")

        # STT, LLM 초기화
        self.stt = STT(openai_api_key=api_key)
        self.extractor = ExtractKeyword(api_key)

        # 오디오 마이크 설정
        mic_config = MicConfig(
            chunk=12000, rate=48000, channels=1,
            record_seconds=5, fmt=pyaudio.paInt16,
            device_index=10, buffer_size=24000,
        )
        self.mic_controller = MicController(mic_config)
        self.wakeup_word = WakeupWord(mic_config.buffer_size)

        # 퍼블리셔
        self.fuel_pub = self.create_publisher(Int32, "/oil", 10)
        self.pump_pub = self.create_publisher(Bool, "/target/Pump", 10)
        self.card_pub = self.create_publisher(Bool, "/target/CreditCard", 10)
        self.detected_tool_pub = self.create_publisher(String, "/detected_tool", 10)  # ← 추가됨

        # 시작 로그
        self.get_logger().info("도구 추출 및 퍼블리시 노드 시작됨")

        # 바로 실행
        self.run_pipeline()

    def run_pipeline(self):
        self.mic_controller.open_stream()
        self.wakeup_word.set_stream(self.mic_controller.stream)
        self.get_logger().info("웨이크업 단어 대기 중...")

        while not self.wakeup_word.is_wakeup():
            pass

        self.get_logger().info("음성 입력 수신 중...")
        
        try:
            text = self.stt.speech2text()
            self.get_logger().info(f"[STT 결과] \"{text}\"")
        except Exception as e:
            self.get_logger().error(f"STT 실패: {e}")
            return
        
        fuel, payment, amount = self.extractor.extract(text)
        self.get_logger().info(f"[LLM 추출 결과] 연료: {fuel}, 결제: {payment}, 금액: {amount}")

        # 연료 퍼블리시

        if fuel in ["휘발유", "경유"]:
            self.pump_pub.publish(Bool(data=True))
            self.detected_tool_pub.publish(String(data="pump"))
        
        if fuel == "경유":
            self.fuel_pub.publish(Int32(data=0))
        elif fuel == "휘발유":
            self.fuel_pub.publish(Int32(data=1))

        # 결제 수단 퍼블리시
        if payment in ["카드", "creditcard", "card"]:
            self.card_pub.publish(Bool(data=True))
            self.detected_tool_pub.publish(String(data="creditcard"))

        # 콘솔 로그
        self.get_logger().info(f"주유 요청: {fuel}, 결제: {payment}, 금액: {amount}")


def main():
    rclpy.init()
    node = ToolPublisherNode()
    rclpy.spin(node)
    node.destroy_node()
    rclpy.shutdown()


if __name__ == "__main__":
    main()


### C-2. 다른 node에서 호출시 동작(get_keyword_custom.py)

설명: [서비스 호출형] 다른 node에서 (/get_keyword) 필요시 호출해서 음성 처리 실행

In [ ]:
import os
import rclpy
import pyaudio
from rclpy.node import Node
from std_srvs.srv import Trigger
from std_msgs.msg import Int32, Bool, String
from dotenv import load_dotenv
# from langchain_community.chat_models import ChatOpenAI
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain
from pathlib import Path
from voice_processing.MicController import MicController, MicConfig
from voice_processing.wakeup_word import WakeupWord
from voice_processing.stt import STT

# === YOLO 클래스 이름과 사용자 표현 매핑 ===
YOLO_CLASS_MAPPING = {
    "creditcard": ["카드", "신용카드", "card"],
    "pump": ["펌프", "주유기", "fuelgun", "건", "주유기기"],
    "cash": ["현금", "돈"]
}

def map_to_yolo_class(keyword: str): # 문자열 비교 기반 매핑
    keyword = keyword.strip().lower()
    for yolo_class, alias_list in YOLO_CLASS_MAPPING.items():
        if keyword in alias_list:
            return yolo_class
    return ""


class ExtractFuelInfo: # 자연어 → [연료 / 결제 / 금액] 추출
    def __init__(self, api_key):
        self.llm = ChatOpenAI(
            model="gpt-4o", temperature=0.2, openai_api_key=api_key
        )

        self.prompt_template = PromptTemplate(
            input_variables=["user_input"],
            template="""
다음 사용자 명령어를 분석해서 [연료 / 결제 / 금액] 형식으로만 결과를 출력하세요.

<제한 사항>
- 출력은 반드시 [연료 / 결제 / 금액] 형식이어야 합니다.
- '[]' 괄호와 '/' 구분자를 포함한 정확한 포맷을 지켜야 합니다.
- 연료: 휘발유 또는 경유
- 결제: 카드 또는 현금
- 금액: 숫자만 (예: "5만원" → 50000)
- 해당 값이 없으면 공백으로 두고 슬래시는 유지합니다.

<예시>
입력: 경유로 5만원 카드 결제해줘 → [경유 / 카드 / 50000]  
입력: 카드 결제할게 → [ / 카드 / ]  
입력: 휘발유로 카드 결제 → [휘발유 / 카드 / ]  

<사용자 입력>
"{user_input}"

<출력 형식>
"""
        )

        self.lang_chain = LLMChain(llm=self.llm, prompt=self.prompt_template)

    def extract(self, text):
        response = self.lang_chain.invoke({"user_input": text})
        try:
            raw = response["text"].strip().strip("[]")
            parts = raw.split("/")
            if len(parts) != 3:
                raise ValueError("응답 형식 오류")
            fuel = parts[0].strip()
            payment = parts[1].strip()
            amount = parts[2].strip()
            return fuel, payment, amount
        except Exception as e:
            print(f"[ERROR] 응답 파싱 실패: {e}")
            return "", "", ""


class AutoFuelNode(Node):
    def __init__(self):
        super().__init__("auto_fuel_node")

        dotenv_path = Path(__file__).parent.parent / "resource" / ".env"
        load_dotenv(dotenv_path=dotenv_path)
        api_key = os.getenv("OPENAI_API_KEY")

        self.stt = STT(openai_api_key=api_key)
        self.extractor = ExtractFuelInfo(api_key)

        mic_config = MicConfig(
            chunk=12000, rate=48000, channels=1,
            record_seconds=5, fmt=pyaudio.paInt16,
            device_index=10, buffer_size=24000,
        )
        self.mic_controller = MicController(mic_config)
        self.wakeup_word = WakeupWord(mic_config.buffer_size)

        self.fuel_pub = self.create_publisher(Int32, "/oil", 10)
        self.creditcard_pub = self.create_publisher(Bool, "/target/CreditCard", 10)
        self.detected_tool_pub = self.create_publisher(String, "/detected_tool", 10)
        # Add amount publisher for gui
        self.amount_pub = self.create_publisher(Int32, "/amount", 10)
        # Add finish
        self.get_logger().info("자동 주유 음성인식 노드 시작됨.")
        self.get_keyword_srv = self.create_service(
            Trigger, "get_keyword", self.get_keyword_callback
        )

    def get_keyword_callback(self, request, response):
        try:
            self.mic_controller.open_stream()
            self.wakeup_word.set_stream(self.mic_controller.stream)
        except OSError:
            self.get_logger().error("오디오 장치 오류")
            response.success = False
            response.message = "오디오 오류"
            return response

        self.get_logger().info("웨이크업 단어 대기 중...")
        while not self.wakeup_word.is_wakeup():
            pass

        self.get_logger().info("음성 수신 중...")
        spoken_text = self.stt.speech2text()

        fuel, payment, amount = self.extractor.extract(spoken_text)
        self.get_logger().info(f"[LLM 추출 결과] 연료: {fuel}, 결제: {payment}, 금액: {amount}")

        # 연료 퍼블리시
        if fuel == "경유":
            self.fuel_pub.publish(Int32(data=0))
        elif fuel == "휘발유":
            self.fuel_pub.publish(Int32(data=1))

        # 결제 수단 퍼블리시
        if payment.lower() in ["카드", "creditcard", "card"]:
            self.creditcard_pub.publish(Bool(data=True))

        # Add amount publish for gui
        if amount.isdigit():
            self.amount_pub.publish(Int32(data=int(amount)))
        # Add finish
        
        # YOLO 타겟 퍼블리시
        fuel_yolo = map_to_yolo_class(fuel)
        payment_yolo = map_to_yolo_class(payment)

        if fuel_yolo:
            self.detected_tool_pub.publish(String(data=fuel_yolo))
        if payment_yolo:
            self.detected_tool_pub.publish(String(data=payment_yolo))

        response.success = True
        response.message = f"{fuel} / {payment} / {amount}"
        return response


def main():
    rclpy.init()
    node = AutoFuelNode()
    rclpy.spin(node)
    node.destroy_node()
    rclpy.shutdown()


if __name__ == "__main__":
    main()


### 두 node의 차이점
<img src="comparison table.png" width="800">

| 항목    | keyword_extraction_3 | get_keyword_custom |
| ----- | -------------------- | ------------------ |
| 실행 방식 | 자동 실행                   | 서비스 호출             |
| 제어 방식   | 사용자 중심                  | 시스템 중심         |
| 제어권   | 없음                   | 있음                 |
| YOLO 매핑 | 단순     | 매핑 함수 있음     |

### D. STT + LLM 결과와 YOLO 객체 인식과 로봇제어 연결(gas_station_yolo_stt_mapping.py)

설명: STT + LLM을 통해 전달된 target 객체를 YOLO로 객체 인식을 수행하고,

RealSense depth 정보를 이용해 객체의 3차원 좌표를 계산하여 로봇 제어 node에 전달하는 node

In [ ]:
# # 1. object detection check - visualization
# # 2. coordination - camera to object x -> baselink to object TF x (calibration -> T_,,,,.npy)
# # 3. 
import rclpy
from rclpy.node import Node
from std_msgs.msg import String
from ultralytics import YOLO
from sensor_msgs.msg import Image, CameraInfo
from cv_bridge import CvBridge
import numpy as np
import time


class ImgNode(Node): # realsence 카메라 이미지 데이터 수신
    def __init__(self):
        super().__init__('img_node')
        self.bridge = CvBridge()
        self.color_frame = None
        self.intrinsics = None

        self.color_subscription = self.create_subscription(
            Image, '/camera/camera/color/image_raw', self.color_callback, 10)

        self.camera_info_subscription = self.create_subscription(
            CameraInfo, '/camera/camera/color/camera_info', self.camera_info_callback, 10)

    def color_callback(self, msg): # RGB 프레임 저장
        self.color_frame = self.bridge.imgmsg_to_cv2(msg, desired_encoding='bgr8')

    def camera_info_callback(self, msg): # 카메라 내부 파라미터 저장
        self.intrinsics = {
            "fx": msg.k[0], "fy": msg.k[4],
            "ppx": msg.k[2], "ppy": msg.k[5]
        }

    def get_color_frame(self): # 최신 RCB 프레임 반환
        return self.color_frame


class YoloTargetFilter(Node):
    def __init__(self): # YOLO 모델 로드, target 토픽 sub, 결과 pub, 카메라 노드 생성
        super().__init__('yolo_target_filter')

        self.target_sub = self.create_subscription(String, '/detected_tool', self.target_callback, 10)
        self.target_keyword = None
        self.publisher_ = self.create_publisher(String, 'yolo_result', 10)

        # YOLO 모델 불러오기
        self.model = YOLO("/home/rokey/Downloads/gasStation_1.pt")
        self.names = self.model.names

        # 내부에서 ImgNode 사용
        self.img_node = ImgNode()

        self.get_logger().info("YOLO 추론 노드 시작 (RealSense 이미지 기반)")

    def _get_depth(self, x, y): # 특정 pixel의 ㅇdepth 값 가져오기
        frame = self._wait_for_valid_data(self.img_node.get_depth_frame, "depth frame")
        try:
            return float(frame[y, x])
        except IndexError:
            self.get_logger().warn(f"Coordinates ({x},{y}) out of range.")
            return None

    def _wait_for_valid_data(self, getter, description): # 카메라 데이터 수신까지 대기
        data = getter()
        while data is None or (isinstance(data, np.ndarray) and not data.any()):
            rclpy.spin_once(self.img_node)
            self.get_logger().info(f"Retry getting {description}.")
            data = getter()
        return data

    def _pixel_to_camera_coords(self, x, y, z): # pixel 자표 -> 3D 좌표로 변환
        fx = self.intrinsics['fx']
        fy = self.intrinsics['fy']
        ppx = self.intrinsics['ppx']
        ppy = self.intrinsics['ppy']
        return (
            (x - ppx) * z / fx,
            (y - ppy) * z / fy,
            z
        )

    def target_callback(self, msg): # STT/LLM target 업데이트
        self.target_keyword = msg.data.lower()
        self.get_logger().info(f"[STT] 타겟 키워드 업데이트: {self.target_keyword}")

    def publish_detection(self, class_name, cx_cal, cy_cal, cz_cal, conf): # 객체 인식 결과 pub
        msg = String()
        msg.data = f"{class_name} {conf:.2f} {cx_cal} {cy_cal} {cz_cal}"
        self.publisher_.publish(msg)
        self.get_logger().info(f"Published: {msg.data}")

    def run(self):
        while rclpy.ok():
            rclpy.spin_once(self.img_node, timeout_sec=0.1)
            frame = self.img_node.get_color_frame()

            if frame is None or self.target_keyword is None:
                continue

            results = self.model.predict(source=frame, conf=0.4, verbose=False, visualize=True)[0]
            
            if results.boxes is None or len(results.boxes) == 0:
                    
                self.get_logger().warn(f"[YOLO] target '{self.target_keyword}' failed detect")
                continue

            for box in results.boxes:
                cx, cy, w, h = map(int, box.xywh[0])
                cz = self._get_depth(cx, cy)
                conf = float(box.conf[0])
                cls_id = int(box.cls[0])
                cls_name = self.names[cls_id].lower()

                if self.intrinsics is None or cz is None:
                    self.get_logger().warn("intrinsics 또는 depth 없음. 건너뜀.")
                    continue

                cx_cal, cy_cal, cz_cal = self._pixel_to_camera_coords(cx, cy, cz)

                if cls_name == self.target_keyword:
                    self.publish_detection(cls_name, cx_cal, cy_cal, cz_cal, conf)


            time.sleep(0.1)


def main():
    rclpy.init()
    node = YoloTargetFilter()
    try:
        node.run()
    finally:
        node.destroy_node()
        rclpy.shutdown()


if __name__ == '__main__':
    main()